# 01 · Datos y calidad

Earth Memory nacional (República Dominicana). Este cuaderno revisa **qué datos entraron, de dónde salieron y con qué cobertura**, antes de mirar cualquier resultado. Solo lee archivos que dejó la corrida (`results/nacional`, `data/nacional/MANIFEST.json`).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
ROOT = Path.cwd()
while not (ROOT / "results" / "nacional").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RES, FIG = ROOT / "results" / "nacional", ROOT / "figures" / "nacional"
corrida = json.loads((RES / "corrida.json").read_text(encoding="utf-8"))
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
def fig(nombre, ancho=900):
    display(Image(filename=str(FIG / f"{nombre}.png"), width=ancho))

## Manifiesto de descargas

Cada archivo descargado tiene su URL exacta (sin credenciales), SHA-256, tamaño y fecha. La sección `rutas` guarda qué ruta funcionó y el error exacto de la que falló.

In [2]:
man = json.loads((ROOT / "data/nacional/MANIFEST.json").read_text(encoding="utf-8"))
arch = pd.DataFrame([{"archivo": k, **{kk: v[kk] for kk in ("bytes", "dataset") if kk in v}} for k, v in man["archivos"].items()])
arch["dataset"] = arch["dataset"].fillna(arch["archivo"].str.split("/").str[0])
display(arch.groupby("dataset").agg(archivos=("archivo", "size"), MB=("bytes", lambda b: round(b.sum() / 1e6, 1))))
pd.DataFrame({k: v for k, v in man.get("rutas", {}).items() if isinstance(v, dict)}).T[["estado", "pedidos", "n_errores"]]

,archivos,MB
dataset,,
GLDAS_M,576,21.9
GRACE_MASCON,1,46.4
IMERG_D,8766,249.2
IMERG_HH,144,3.5
IMERG_M,288,7.2
gldas,1,0.9
limites,2,0.1
natural_earth_10m_DOM_extracto.geojson,1,1.0
power,21,3.2


,estado,pedidos,n_errores
IMERG_M,ok,288,0
GLDAS_M,ok,288,0
IMERG_HH,ok,144,0
GLDAS_RAIN,ok,288,0
IMERG_D,ok,8766,0
GRACE,ok,1,0


## Celdas: cuántas hay y cuántas entran

Una celda entra si tiene al menos 25 % de su área en RD, más RD que Haití y (en GLDAS) dato de suelo. Se informa también cuántas tocan el país.

In [3]:
pd.Series({k: v for k, v in corrida.items() if k.startswith("celdas_") or k.startswith("dias_") or k.startswith("meses_")})

celdas_imerg_caja                 1014
celdas_imerg_incluidas             442
celdas_imerg_tocan_pais            489
celdas_gldas_caja                  176
celdas_gldas_incluidas              65
celdas_gldas_tocan_pais            100
celdas_gldas_tierra_tocan_pais      72
meses_imerg_faltantes               []
dias_imerg                        8766
dias_imerg_faltantes                 0
dtype: object

In [4]:
ci = pd.read_csv(RES / "celdas_imerg.csv"); cg = pd.read_csv(RES / "celdas_gldas.csv")
fig_, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, c, t in ((ax[0], ci, "IMERG 0.1°"), (ax[1], cg, "GLDAS 0.25°")):
    s = a.scatter(c["lon"], c["lat"], c=c["frac_pais"], s=6 if t.startswith("IMERG") else 30, marker="s", cmap="viridis")
    a.scatter(c.loc[c["incluida"], "lon"], c.loc[c["incluida"], "lat"], s=1, color="k")
    a.set_title(f"{t}: fracción en RD (punto negro = incluida)", fontsize=9); a.set_aspect(1 / np.cos(np.radians(18.8)))
plt.colorbar(s, ax=ax, shrink=0.8); plt.show()
cg["motivo_exclusion"].fillna("incluida").value_counts()

motivo_exclusion
GLDAS la trata como mar; fracción en DOM 0.00 < 0.25                      69
incluida                                                                  65
GLDAS la trata como mar                                                   10
fracción en DOM 0.00 < 0.25; dominada por HTI                              7
GLDAS la trata como mar; fracción en DOM 0.04 < 0.25                       3
GLDAS la trata como mar; fracción en DOM 0.25 < 0.25                       2
GLDAS la trata como mar; fracción en DOM 0.00 < 0.25; dominada por HTI     2
GLDAS la trata como mar; fracción en DOM 0.08 < 0.25                       2
GLDAS la trata como mar; fracción en DOM 0.03 < 0.25                       2
dominada por HTI                                                           2
GLDAS la trata como mar; fracción en DOM 0.17 < 0.25                       2
GLDAS la trata como mar; fracción en DOM 0.11 < 0.25                       1
GLDAS la trata como mar; fracción en DOM 0.05 < 0.25; domin

## Provincias (geoBoundaries ADM1)

Cuántas celdas aportan a cada provincia. Provincias chicas (p. ej. Distrito Nacional) dependen de muy pocas celdas GLDAS: sus series de suelo son casi las de 1-2 celdas.

In [5]:
pd.read_csv(RES / "provincias_cobertura.csv").sort_values("area_km2")

,provincia,celdas_imerg,area_km2,celdas_gldas
4,Distrito Nacional,3,78.9909,1
9,Hermanas,10,481.0440,5
13,La Romana,9,520.2850,3
31,Valverde,14,765.2880,4
20,Peravia,11,767.7740,3
24,San José de Ocoa,14,840.5890,6
7,Espaillat,17,865.8380,5
22,Samaná,19,968.8000,3
16,Monseñor Nouel,16,978.9150,6
28,Santiago Rodríguez,19,1103.9800,7


## Lluvia IMERG frente a la lluvia que forzó a GLDAS

GLDAS-2.1 se forzó con GPCP, no con IMERG. Si las dos lluvias no coinciden, una 'contradicción' entre lluvia IMERG y suelo GLDAS puede ser diferencia entre productos. Aquí se comparan los SPI-3 de las dos en cada celda GLDAS.

In [6]:
p = pd.read_parquet(RES / "parquet/panel_celdas_gldas_mensual.parquet")
spi = p[p["variable"] == "SPI3"].pivot(index="fecha", columns="celda", values="valor")
ep_mod = RES / "episodios_lluvia_modelo.csv"
print("Episodios con lluvia del modelo:", len(pd.read_csv(ep_mod)) if ep_mod.exists() else "no disponible")
ct = pd.read_csv(RES / "contradicciones_por_celda.csv")
cols = [c for c in ct.columns if c.startswith("n_A_raiz") or c.startswith("n_episodios")]
ct[cols].sum()

Episodios con lluvia del modelo: 806


n_episodios                  813
n_A_raiz                      96
n_episodios_lluvia_modelo    806
n_A_raiz_lluvia_modelo        68
dtype: int64

## Comprobación con NASA POWER (MERRA-2)

POWER sirve MERRA-2 a ~0.5°. No es independiente para el suelo (los dos son modelos), pero sí ayuda a ver si la lluvia y la raíz se mueven igual.

In [7]:
f = RES / "comparacion_power_merra2.csv"
pd.read_csv(f).describe().T[["count", "mean", "50%", "min", "max"]] if f.exists() else "sin datos POWER"

,count,mean,50%,min,max
celdas_imerg,18.0,18.777778,20.000000,5.000000,28.000000
celdas_gldas,18.0,3.444444,3.500000,1.000000,6.000000
r_spi3_imerg_vs_merra2,18.0,0.421936,0.428214,0.251257,0.522823
r_z_raiz_gldas_vs_merra2,18.0,0.149373,0.154164,-0.013766,0.383890
lluvia_anual_imerg_mm,18.0,1460.008333,1516.995000,1016.000000,1835.720000
lluvia_anual_merra2_mm,18.0,714.354222,694.058000,554.573000,1086.470000
